In [4]:
import os
from dotenv import load_dotenv
load_dotenv()
groq_api_key= os.getenv("GROQ_API_KEY")

In [17]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [21]:
from langchain_huggingface import HuggingFaceEmbeddings

In [22]:
from langchain_chroma import Chroma

In [24]:
from langchain_groq import ChatGroq

In [29]:
loader = TextLoader("lg_t&c.txt")
doc = loader.load()


In [31]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=400,chunk_overlap=20)

In [73]:
split = text_splitter.split_documents(doc)
split

[Document(metadata={'source': 'lg_t&c.txt'}, page_content='TERMS AND CONDITIONS OF PURCHASE\n\n \n\nLast Updated: May 8, 2025\n \n\nSummary\n\nThis summary is provided only for convenience. This summary is not a replacement for reading and understanding the terms and conditions. In case of any conflict between this summary and any other part of the terms, such other part of the terms shall govern.'),
 Document(metadata={'source': 'lg_t&c.txt'}, page_content='At https://www.lg.com/in you can purchase products for delivery only in India. Use of this service is subject to the following conditions, among others:\n\n• You must be at least 18 years old and competent to enter into a contract to use this service;\n\n• Abusive or illegal behaviour will not be tolerated;'),
 Document(metadata={'source': 'lg_t&c.txt'}, page_content='If you do any or all of the following, you indicate that you have read, understood, and agree to the terms and conditions:\n\n• Explicitly accepting the terms and con

In [32]:
embedder = HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [38]:
from langchain_community.vectorstores import FAISS

In [74]:
vectorstore = FAISS.from_documents(split,embedder)
vectorstore

In [75]:
vectorstore.similarity_search("lg")

[Document(id='8e7e17f9-981e-4ba8-8003-49fd03ecc711', metadata={'source': 'lg_t&c.txt'}, page_content='please visit www.lg.com/in or contact retailer / store manager.'),
 Document(id='f75cf854-6572-4c1c-8b40-b19e956bb674', metadata={'source': 'lg_t&c.txt'}, page_content='LG Electronics India Limited (“LGE”), along with its group companies (“we” or “us”), is happy to provide services to you (the “User or “you””). These Terms of Service explain how you can use LG’s services, including:'),
 Document(id='7300e068-294e-4b1c-ba6e-8701f1576cab', metadata={'source': 'lg_t&c.txt'}, page_content='This Offer is brought to you by LG Electronics India Limited (“LG” or “LGEIL”). By participating in this offer, the Participant/user confirms that they have read, understood, and accepted these Terms and Conditions and agrees to be bound by all terms and conditions outlined herein. Participation in this Offer is voluntary and subject to these Terms and Conditions, the terms of use of'),
 Document(id='10c

In [76]:
retriever = vectorstore.as_retriever(search_kwargs = {"k":3})

In [44]:
llm = ChatGroq(model="openai/gpt-oss-120b")

In [48]:
from langchain_core.prompts import ChatPromptTemplate

In [77]:
system_prompts = """ you are expert support agent and answer the user query to best fit answer.
                   in no condition you will answer from out of  context 
                    answer only from the uploaded pdf or txt file 
                   if there is no context of question answer not enough context/ information in this file 
                 
                 Context:{context}"""

prompts=ChatPromptTemplate.from_messages(
    [ 
        ("system",system_prompts),
        ("human","{input}"),
         ]
)

In [61]:
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [78]:
question_answer_chain=create_stuff_documents_chain(llm,prompts)
rag_chain = create_retrieval_chain(retriever,question_answer_chain)


In [79]:
response= rag_chain.invoke({"input" :" what is lg"})

In [81]:
print(response["answer"])

LG refers to **LG Electronics India Limited** (often abbreviated as “LGE”) and its group of companies. In the provided terms of service, LG is described as the provider of various services—including the LG website, the ThinQ mobile app, and related product features—offered to users.


In [82]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
